# Gradient Artifact `t0` Detection and Synchronization Demo

Este notebook muestra como estimar `t0` directamente desde la morfologia de los artefactos de gradient en una senal EEG simultanea EEG-fMRI, sin usar trigger dedicado. Despues alinea la senal al inicio estimado y deja el resultado listo para limpiar GA con el pipeline existente.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np

START_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (START_DIRECTORY, *START_DIRECTORY.parents) if (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root containing src/.")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from functions.aas_ga import run_aas_pipeline
from functions.artifind_ga import detect_artifind_gradient_onsets
from functions.gradient_sync import detect_gradient_artifact_start
from functions.load_fmri_metadata import load_fmri_metadata


In [ ]:
EEG_ROOT = PROJECT_ROOT / "data" / "raw" / "Dataset1" / "Simultaneous_EEG_fMRI" / "BIDS_dataset_EEG"
FMRI_ROOT = PROJECT_ROOT / "data" / "raw" / "Dataset1" / "Simultaneous_EEG_fMRI" / "BIDS_dataset_MRI"

SUBJECT = "sub-007"
EEG_TASK = "fmrirestingec"
FMRI_TASK = "rest"

CALIBRATION_SECONDS = 5.0
THRESHOLD_SIGMA = 3.0
MIN_CHANNELS_FRACTION = 0.05
REFRACTORY_MS = 10.0
CLUSTER_TOLERANCE_MS = 12.0
REFERENCE_CHANNEL = 0
MAX_CHANNELS_TO_PLOT = 3
PLOT_WINDOW_SECONDS = 4.0
ARTIFIND_DUMMY_SCANS = 0
ARTIFIND_TRIGGER_MODE = "auto"

def get_eeg_set_path(subject: str, task: str = EEG_TASK, eeg_root: Path = EEG_ROOT) -> Path:
    eeg_path = eeg_root / subject / "eeg" / f"{subject}_task-{task}_eeg.set"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    return eeg_path


def get_fmri_nifti_path(subject: str, task: str = FMRI_TASK, fmri_root: Path = FMRI_ROOT) -> Path:
    fmri_path = fmri_root / subject / "ses-001" / "func" / f"{subject}_ses-001_task-{task}_bold.nii.gz"
    if not fmri_path.exists():
        raise FileNotFoundError(f"fMRI file not found: {fmri_path}")
    return fmri_path


def load_raw_eeg(eeg_path: Path) -> mne.io.BaseRaw:
    return mne.io.read_raw_eeglab(eeg_path, preload=True, verbose="ERROR")


def select_analysis_channels(raw: mne.io.BaseRaw) -> tuple[list[int], list[str]]:
    excluded_tokens = ("ECG", "EKG", "VREF", "TRIG", "STI", "MISC", "RESP", "EOG", "EMG", "AUX")
    picks: list[int] = []
    names: list[str] = []
    for index, channel_name in enumerate(raw.ch_names):
        upper_name = channel_name.upper()
        if any(token in upper_name for token in excluded_tokens):
            continue
        picks.append(index)
        names.append(channel_name)
    if not picks:
        picks = list(range(len(raw.ch_names)))
        names = list(raw.ch_names)
    return picks, names


## Load data

El ejemplo usa un sujeto real del dataset, pero puedes cambiar `SUBJECT` y `EEG_TASK` para probar otra sesion. El dataset no expone un trigger dedicado en EEG, asi que el inicio de la secuencia fMRI se estima por la morfologia de los artefactos.


In [ ]:
eeg_path = get_eeg_set_path(SUBJECT)
fmri_path = get_fmri_nifti_path(SUBJECT)

raw = load_raw_eeg(eeg_path)
fmri_info = load_fmri_metadata(fmri_path)

analysis_picks, analysis_names = select_analysis_channels(raw)
eeg_signal = raw.get_data(picks=analysis_picks)
fs = float(raw.info["sfreq"])

print(f"Subject: {SUBJECT}")
print(f"EEG task: {EEG_TASK}")
print(f"EEG path: {eeg_path}")
print(f"fMRI path: {fmri_path}")
print(f"EEG shape: {eeg_signal.shape}")
print(f"Sampling frequency: {fs} Hz")
print(f"Selected channels: {len(analysis_picks)} / {len(raw.ch_names)}")
print(f"fMRI TR: {fmri_info['TR_s']} s")
print(f"fMRI TE: {fmri_info['TE_s']} s")
print(f"fMRI slices: {fmri_info['n_slices']}")
print(f"fMRI volumes: {fmri_info['n_volumes']}")
print(f"fMRI matrix XY: {fmri_info['matrix_xy']}")
print("Dynamic range note: the dataset metadata available here does not expose the EEG hardware input range, so ±65.5 mV cannot be confirmed from the files alone.")


## Artifind detection and comparison

This section runs the independent Artifind-style implementation on the same EEG channels. The experiment assumes zero dummy scans. `meets_artifind_criteria` must be checked: if it is false, the expected onset count was not met within the published ±7 tolerance and the code returns the closest periodic candidate only because `allow_approximate=True`. Independently of that global criterion, `t0` must start a locally periodic GA grid over 2 TR; isolated earlier peaks are discarded.


In [ ]:
detection = detect_gradient_artifact_start(
    eeg_signal,
    fs=fs,
    tr_sec=float(fmri_info["TR_s"]),
    n_slices=int(fmri_info["n_slices"]),
    channel_names=analysis_names,
    calibration_seconds=CALIBRATION_SECONDS,
    threshold_sigma=THRESHOLD_SIGMA,
    min_channels_fraction=MIN_CHANNELS_FRACTION,
    refractory_ms=REFRACTORY_MS,
    cluster_tolerance_ms=CLUSTER_TOLERANCE_MS,
    min_cycle_peaks=3,
)

artifind_detection = detect_artifind_gradient_onsets(
    eeg_signal,
    fs=fs,
    tr_sec=float(fmri_info["TR_s"]),
    n_volumes=int(fmri_info["n_volumes"]),
    n_slices=int(fmri_info["n_slices"]),
    n_dummy_scans=ARTIFIND_DUMMY_SCANS,
    channel_names=analysis_names,
    trigger_mode=ARTIFIND_TRIGGER_MODE,
    allow_approximate=True,
)

print("GradientSync (existing detector)")
print(f"  T_GA coarse:{detection.t_ga_coarse_sample:8d} samples | {detection.t_ga_coarse_sec:10.6f} s")
print(f"  T_GA:       {detection.t_ga_sample:8d} samples | {detection.t_ga_sec:10.6f} s")
print(f"  T_BOLD/t0:  {detection.t_bold_sample:8d} samples | {detection.t_bold_sec:10.6f} s")
print(f"  T_BOLD valid: {detection.t_bold_valid}; grid match: {detection.t_bold_match_fraction:.1%}")
print(f"  T_F peak:    {detection.t_f_sample:8d} samples | {detection.t_f_sec:10.6f} s")
print(f"  T_F end:     {detection.t_f_end_sample:8d} samples | {detection.t_f_end_sec:10.6f} s")
print(f"  T_F valid: {detection.t_f_valid}; reverse grid match: {detection.t_f_match_fraction:.1%}")
print(f"  T_F method: {detection.t_f_method}")
print(f"  Later isolated peaks discarded for T_F: {detection.t_f_discarded_trailing_peaks}")
print("Artifind-style detector")
print(f"  T0/first GA:{artifind_detection.t0_sample:8d} samples | {artifind_detection.t0_sec:10.6f} s")
print(f"  Trigger type: {artifind_detection.trigger_type}")
print(f"  Selected channel: {artifind_detection.selected_channel_name}")
print(f"  GA onsets: {artifind_detection.detected_onset_count} / {artifind_detection.expected_onset_count} expected")
print(f"  Interval mode: {artifind_detection.interval_mode_samples} samples; expected {artifind_detection.expected_period_samples:.6f}")
print(f"  Meets published period/count criteria: {artifind_detection.meets_artifind_criteria}")
print(f"  Local onset grid valid: {artifind_detection.local_onset_valid}; match over 2 TR: {artifind_detection.local_match_fraction:.1%}")
print(f"  Earlier isolated peaks discarded: {artifind_detection.discarded_leading_onsets}")
print(f"  T_F peak:    {artifind_detection.t_f_sample:8d} samples | {artifind_detection.t_f_sec:10.6f} s")
print(f"  T_F end:     {artifind_detection.t_f_end_sample:8d} samples | {artifind_detection.t_f_end_sec:10.6f} s")
print(f"  T_F valid: {artifind_detection.t_f_valid}; nearest-grid error: {artifind_detection.t_f_error_samples:.3f} samples")
print(f"  Later peaks ignored for T_F: {artifind_detection.discarded_trailing_onsets}")
print("Differences")
print(f"  Artifind - T_GA:   {artifind_detection.t0_sec - detection.t_ga_sec:+.6f} s")
print(f"  Artifind - T_BOLD: {artifind_detection.t0_sec - detection.t_bold_sec:+.6f} s")
print(f"  Artifind - T_F peak: {artifind_detection.t_f_sec - detection.t_f_sec:+.6f} s")
print(f"  Artifind - T_F end:  {artifind_detection.t_f_end_sec - detection.t_f_end_sec:+.6f} s")


### Visual comparison around the detected starts

The raw EEG is shown with all three checkpoints, following the same visual style as the GradientSync example below. Blue circles mark the Artifind GA peaks visible inside the displayed interval.


In [ ]:
start_samples = np.array([detection.t_ga_sample, detection.t_bold_sample, artifind_detection.t0_sample])
start_separation_sec = (start_samples.max() - start_samples.min()) / fs
comparison_window_seconds = 0.25 if start_separation_sec <= 1.0 else PLOT_WINDOW_SECONDS
window_samples = int(round(comparison_window_seconds * fs))
artifind_channel = artifind_detection.selected_channel_index
comparison_channels = [artifind_channel]
comparison_channels.extend(index for index in range(eeg_signal.shape[0]) if index != artifind_channel)
comparison_channels = comparison_channels[:min(MAX_CHANNELS_TO_PLOT, eeg_signal.shape[0])]

early_start = max(0, min(detection.t_ga_sample, artifind_detection.t0_sample) - window_samples)
early_stop = min(eeg_signal.shape[1], max(detection.t_ga_sample, artifind_detection.t0_sample) + window_samples)
bold_start = max(0, detection.t_bold_sample - window_samples)
bold_stop = min(eeg_signal.shape[1], detection.t_bold_sample + window_samples)
comparison_windows = [("T_GA and Artifind t0", early_start, early_stop)]
if bold_start > early_stop or bold_stop < early_start:
    comparison_windows.append(("GradientSync T_BOLD/t0", bold_start, bold_stop))

fig, axes = plt.subplots(
    len(comparison_channels), len(comparison_windows),
    figsize=(7 * len(comparison_windows), 2.7 * len(comparison_channels)),
    squeeze=False,
)
markers = [
    (detection.t_ga_coarse_sample, detection.t_ga_coarse_sec, "purple", ":", "GradientSync T_GA coarse"),
    (detection.t_ga_sample, detection.t_ga_sec, "crimson", "--", "GradientSync T_GA"),
    (detection.t_bold_sample, detection.t_bold_sec, "darkorange", "--", "GradientSync T_BOLD/t0"),
    (artifind_detection.t0_sample, artifind_detection.t0_sec, "tab:blue", "-.", "Artifind t0"),
]

for row, channel_index in enumerate(comparison_channels):
    for column, (window_title, window_start, window_stop) in enumerate(comparison_windows):
        axis = axes[row, column]
        window_time = np.arange(window_start, window_stop) / fs
        axis.plot(window_time, eeg_signal[channel_index, window_start:window_stop], color="0.25", linewidth=0.8)
        for marker_sample, marker_sec, color, linestyle, label in markers:
            if window_start <= marker_sample < window_stop:
                axis.axvline(marker_sec, color=color, linestyle=linestyle, linewidth=1.4, label=label)
        visible_onsets = artifind_detection.ga_onsets[
            (artifind_detection.ga_onsets >= window_start)
            & (artifind_detection.ga_onsets < window_stop)
        ]
        if visible_onsets.size:
            axis.scatter(
                visible_onsets / fs, eeg_signal[channel_index, visible_onsets],
                color="tab:blue", marker="o", s=20, zorder=3, label="Artifind GA onsets",
            )
        if row == 0:
            axis.set_title(window_title)
        if column == 0:
            axis.set_ylabel(analysis_names[channel_index])
        if row == len(comparison_channels) - 1:
            axis.set_xlabel("Time (s)")
        axis.grid(alpha=0.2)
        if row == 0:
            axis.legend(loc="upper right", fontsize=8)

fig.suptitle("GradientSync vs. Artifind: t0 and gradient-artifact onsets")
fig.tight_layout()
plt.show()


### Visual comparison around the detected GA ends

`T_F peak` marks the final validated GA peak. `T_F end` adds one slice/volume period and estimates the end of the final acquisition cycle. When both algorithms are within one second, the plot automatically uses a ±0.25 s detail window.


In [ ]:
end_separation_sec = abs(detection.t_f_sec - artifind_detection.t_f_sec)
if end_separation_sec <= 1.0:
    end_half_window_samples = int(round(0.25 * fs))
    end_windows = [(
        "GradientSync and Artifind T_F",
        max(0, min(detection.t_f_sample, artifind_detection.t_f_sample) - end_half_window_samples),
        min(eeg_signal.shape[1], max(detection.t_f_end_sample, artifind_detection.t_f_end_sample) + end_half_window_samples),
    )]
else:
    end_half_window_samples = int(round(PLOT_WINDOW_SECONDS * fs))
    end_windows = [
        ("GradientSync T_F", max(0, detection.t_f_sample - end_half_window_samples), min(eeg_signal.shape[1], detection.t_f_end_sample + end_half_window_samples)),
        ("Artifind T_F", max(0, artifind_detection.t_f_sample - end_half_window_samples), min(eeg_signal.shape[1], artifind_detection.t_f_end_sample + end_half_window_samples)),
    ]

end_markers = [
    (detection.t_ga_end_sample, detection.t_ga_end_sec, "seagreen", ":", "GradientSync energy end"),
    (detection.t_f_sample, detection.t_f_sec, "crimson", "--", "GradientSync T_F peak"),
    (detection.t_f_end_sample, detection.t_f_end_sec, "darkorange", "--", "GradientSync T_F end"),
    (artifind_detection.t_f_sample, artifind_detection.t_f_sec, "tab:blue", "-.", "Artifind T_F peak"),
    (artifind_detection.t_f_end_sample, artifind_detection.t_f_end_sec, "tab:cyan", "-.", "Artifind T_F end"),
]

fig, axes = plt.subplots(
    len(comparison_channels), len(end_windows),
    figsize=(7 * len(end_windows), 2.7 * len(comparison_channels)),
    squeeze=False,
)
for row, channel_index in enumerate(comparison_channels):
    for column, (window_title, window_start, window_stop) in enumerate(end_windows):
        axis = axes[row, column]
        window_time = np.arange(window_start, window_stop) / fs
        axis.plot(window_time, eeg_signal[channel_index, window_start:window_stop], color="0.25", linewidth=0.8)
        for marker_sample, marker_sec, color, linestyle, label in end_markers:
            if window_start <= marker_sample < window_stop:
                axis.axvline(marker_sec, color=color, linestyle=linestyle, linewidth=1.4, label=label)
        visible_onsets = artifind_detection.ga_onsets[(artifind_detection.ga_onsets >= window_start) & (artifind_detection.ga_onsets < window_stop)]
        if visible_onsets.size:
            axis.scatter(visible_onsets / fs, eeg_signal[channel_index, visible_onsets], color="tab:blue", s=20, zorder=3, label="Artifind GA onsets")
        if row == 0:
            axis.set_title(window_title)
            axis.legend(loc="upper right", fontsize=8)
        if column == 0:
            axis.set_ylabel(analysis_names[channel_index])
        if row == len(comparison_channels) - 1:
            axis.set_xlabel("Time (s)")
        axis.grid(alpha=0.2)
fig.suptitle("GradientSync vs. Artifind: final GA peak and estimated acquisition end")
fig.tight_layout()
plt.show()


## GradientSync diagnostics

The existing detector result computed in the comparison above uses a calibration window to estimate a robust threshold, applies peak thinning, merges detections across channels, and validates periodicity against the expected `TR / n_slices` spacing.


In [ ]:
print(f"T_GA coarse sample: {detection.t_ga_coarse_sample}")
print(f"T_GA coarse time: {detection.t_ga_coarse_sec:.6f} s")
print(f"T_GA sample: {detection.t_ga_sample}")
print(f"T_GA time: {detection.t_ga_sec:.6f} s")
print(f"T_GA end: {detection.t_ga_end_sec:.6f} s")
print(f"T_BOLD sample: {detection.t_bold_sample}")
print(f"T_BOLD time: {detection.t_bold_sec:.6f} s")
print(f"T_BOLD end: {detection.t_bold_end_sec:.6f} s")
print(f"T_BOLD valid: {detection.t_bold_valid}")
print(f"T_BOLD grid match: {detection.t_bold_match_fraction:.1%}")
print(f"Threshold: {detection.threshold:.6f}")
print(f"Calibration seconds used: {detection.calibration_seconds:.1f}")
print(f"Slice period: {detection.slice_period_sec:.6f} s")
print(f"Consensus peaks: {detection.consensus_peaks.size}")
print(f"Min consensus votes: {int(detection.consensus_votes.min())}")


## Visual check around `t0`

This plot shows a short window around the estimated onset. The red line marks the first detected RF pulse that defines `t0`.


In [ ]:
energy_time = detection.energy_samples / fs
plt.figure(figsize=(15, 4))
plt.plot(energy_time, detection.multichannel_energy, color='tab:purple', lw=0.9, label='Energia alta frecuencia multicanal')
plt.axhline(detection.energy_threshold, color='black', ls=':', label='Umbral de GA sostenido')
plt.axvline(detection.t_ga_coarse_sec, color='purple', ls=':', label='T_GA coarse')
plt.axvline(detection.t_ga_sec, color='crimson', ls='--', label='T_GA refined')
plt.axvline(detection.t_bold_sec, color='darkorange', ls='--', label='T_BOLD')
plt.axvline(detection.t_ga_end_sec, color='seagreen', ls='--', label='Fin T_GA')
plt.xlabel('Tiempo (s)'); plt.ylabel('Energia normalizada')
plt.title('Diagnostico de deteccion GA/BOLD')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
window_samples = int(round(PLOT_WINDOW_SECONDS * fs))
start = max(0, detection.t_ga_sample - window_samples)
stop = min(eeg_signal.shape[1], detection.t_bold_sample + window_samples)
time_axis = np.arange(start, stop) / fs

n_plot = min(MAX_CHANNELS_TO_PLOT, eeg_signal.shape[0])
fig, axes = plt.subplots(n_plot, 1, figsize=(14, 2.5 * n_plot), sharex=True)
if n_plot == 1:
    axes = [axes]

for axis, channel_index in zip(axes, range(n_plot), strict=False):
    axis.plot(time_axis, eeg_signal[channel_index, start:stop], linewidth=0.8)
    axis.axvline(detection.t_ga_sec, color="crimson", linestyle="--", linewidth=1.2, label="T_GA")
    axis.axvline(detection.t_bold_sec, color="darkorange", linestyle="--", linewidth=1.2, label="T_BOLD")
    axis.set_ylabel(analysis_names[channel_index])

axes[-1].set_xlabel("Time (s)")
axes[0].legend(loc="upper right")
fig.suptitle("EEG around T_GA and T_BOLD")
fig.tight_layout()


## Synchronize and prepare for GA cleaning

The simplest synchronization is to re-reference time so that the first detected RF pulse becomes `t = 0`. The same `t0` and `t_f` can then be used to crop the fMRI interval and pass the onset as the `offset` for the existing trigger-free AAS pipeline.


In [ ]:
aligned_signal = eeg_signal[:, detection.t_ga_sample:detection.t_ga_end_sample]
aligned_time = np.arange(aligned_signal.shape[1]) / fs
print(f"GA interval: {detection.t_ga_sec:.6f} s -> {detection.t_ga_end_sec:.6f} s")
print(f"Stable BOLD interval: {detection.t_bold_sec:.6f} s -> {detection.t_bold_end_sec:.6f} s")

aas_result = run_aas_pipeline(
    aligned_signal,
    TR=float(fmri_info["TR_s"]),
    fs=fs,
    reference_channel=REFERENCE_CHANNEL,
    offset=0,
    window_size=21,
)

cleaned_signal = aas_result["cleaned_signal"]
print(f"Aligned signal shape: {aligned_signal.shape}")
print(f"Cleaned signal shape: {cleaned_signal.shape}")
print(f"AAS offset used: {aas_result['offset']}")
print(f"TR samples: {aas_result['T_samples']}")
print(f"First segment lag estimates: {aas_result['lags'][:10]}")


## MNE visualization

The same aligned interval can be opened with the MNE browser, matching the style used in the other GA notebooks.


In [ ]:
mne.viz.set_browser_backend("qt")

mne_info = mne.create_info(ch_names=analysis_names, sfreq=fs, ch_types=["eeg"] * len(analysis_names))
aligned_raw_mne = mne.io.RawArray(aligned_signal, mne_info.copy(), verbose="ERROR")
cleaned_raw_mne = mne.io.RawArray(cleaned_signal, mne_info.copy(), verbose="ERROR")

print("Aligned EEG window in MNE browser")
aligned_raw_mne.plot(n_channels=min(20, len(analysis_names)), duration=10, show_scrollbars=True, block=True)

print("Cleaned EEG window in MNE browser")
cleaned_raw_mne.plot(n_channels=min(20, len(analysis_names)), duration=10, show_scrollbars=True, block=True)
